# Governance Officer Analysis
## Table of Contents

1. [PII Identification](#1-pii-identification)
   - 1.1 [Sensitive Columns Check](#11-sensitive-columns-check)
2. [Pseudonymization Demonstration](#2-pseudonymization-demonstration)
   - 2.1 [Email Hashing with Salt](#21-email-hashing-with-salt)
   - 2.2 [SSN Tokenization with UUID](#22-ssn-tokenization-with-uuid)
   - 2.3 [Sanitized Dataset](#23-sanitized-dataset)
3. [GDPR Compliance Analysis](#3-gdpr-compliance-analysis)
   - 3.1 [Article 5(1)(c) – Data Minimisation](#31-article-51c-data-minimisation)
   - 3.2 [Article 5(1)(e) – Storage Limitation](#32-article-51e-storage-limitation)
   - 3.3 [Article 6 – Lawfulness of Processing](#33-article-6-lawfulness-of-processing)
   - 3.4 [Article 17 – Right to Erasure](#34-article-17-right-to-erasure)
   - 3.5 [Article 32 – Security of Processing](#35-article-32-security-of-processing)
4. [EU AI Act Analysis](#4-eu-ai-act-analysis)
   - 4.1 [Classification](#41-classification)
   - 4.2 [Requirements for High-Risk AI Systems](#42-requirements-for-high-risk-ai-systems)
   - 4.3 [Consequences of Non-Compliance](#43-consequences-of-non-compliance)
   - 4.4 [Summary](#44-Summary)
5. [Governance Recommendations](#5-governance-recommendations)
   - 5.1 [Immediate Actions (Short-term)](#51-immediate-actions-short-term)
   - 5.2 [Medium-term Actions](#52-medium-term-actions)
   - 5.3 [Long-term Actions](#53-long-term-actions)
   - 5.4 [Technical Implementation Priorities](#54-technical-implementation-priorities)


In [1]:
# Import libraries
import pandas as pd
import hashlib
import numpy as np
from datetime import datetime


print("LOADING CLEANED CREDIT APPLICATIONS DATA")


# Load the cleaned CSV file (created by the Data Engineer)
# The file is located in the data folder (one level up from notebooks)
df = pd.read_csv('../data/credit_applications_clean.csv')

print(f"\n Data loaded successfully!")
print(f"   - Shape: {df.shape}")
print(f"   - Rows: {df.shape[0]}")
print(f"   - Columns: {df.shape[1]}")
print(f"\nFirst 5 rows:")
print(df.head())


print("COLUMN NAMES AND DATA TYPES")

print(df.dtypes)

LOADING CLEANED CREDIT APPLICATIONS DATA

 Data loaded successfully!
   - Shape: (501, 20)
   - Rows: 501
   - Columns: 20

First 5 rows:
       _id                                  spending_behavior  \
0  app_200  [{'category': 'Shopping', 'amount': 480}, {'ca...   
1  app_037  [{'category': 'Rent', 'amount': 608}, {'catego...   
2  app_215              [{'category': 'Rent', 'amount': 109}]   
3  app_024           [{'category': 'Fitness', 'amount': 575}]   
4  app_184     [{'category': 'Entertainment', 'amount': 463}]   

        processing_timestamp applicant_info.full_name  \
0  2024-01-15 00:00:00+00:00              Jerry Smith   
1                        NaN           Brandon Walker   
2                        NaN              Scott Moore   
3                        NaN               Thomas Lee   
4  2024-01-15 00:00:00+00:00          Brian Rodriguez   

         applicant_info.email applicant_info.ssn applicant_info.ip_address  \
0   jerry.smith17@hotmail.com        596-64-4340  

## 1. PII Identification (Personal Identifiable Information)

According to GDPR (Article 4), PII is any information that can directly or indirectly identify a natural person.

In the NovaCred dataset, the following PII fields were identified:

| Field | Data Type | Risk Level | Justification |
|-------|-----------|------------|---------------|
| *applicant_info.full_name* | String | High | Direct identification |
| *applicant_info.email* | String | High | Direct identification |
| *applicant_info.ssn* | String | **Critical** | Social Security Number (unique identifier, extremely sensitive) |
| *applicant_info.ip_address* | String | Medium | Can be used to derive location or identity |
| *applicant_info.date_of_birth* | String | Medium | When combined with other data, can identify |

**Note:** The Data Engineer removed columns with >95% missing values, but **sensitive columns (SSN, IP) are still present** in this cleaned version. This represents a privacy risk that needs to be addressed.

### 1.1 Sensitive Columns check

In [2]:

print("CHECKING SENSITIVE COLUMNS")


# Check if SSN column exists
if 'applicant_info.ssn' in df.columns:
    print(f"\n Column 'applicant_info.ssn' exists!")
    print(f"   - Number of non-null SSNs: {df['applicant_info.ssn'].notna().sum()}")
    print(f"   - Sample values: {df['applicant_info.ssn'].head(3).tolist()}")
else:
    print("\n Column 'applicant_info.ssn' NOT found (good for privacy)")

# Check if IP address column exists
if 'applicant_info.ip_address' in df.columns:
    print(f"\n Column 'applicant_info.ip_address' exists!")
    print(f"   - Number of non-null IPs: {df['applicant_info.ip_address'].notna().sum()}")
    print(f"   - Sample values: {df['applicant_info.ip_address'].head(3).tolist()}")
else:
    print("\n Column 'applicant_info.ip_address' NOT found (good for privacy)")

# Check if email column exists (also PII)
if 'applicant_info.email' in df.columns:
    print(f"\n Column 'applicant_info.email' exists (PII)")
    print(f"   - Number of non-null emails: {df['applicant_info.email'].notna().sum()}")
    
    # Check for invalid emails (missing @ symbol)
    invalid_emails = df[~df['applicant_info.email'].str.contains('@', na=False)]['applicant_info.email'].count()
    print(f"   - Emails without '@' symbol: {invalid_emails}")
    
    # Show examples of invalid emails if any
    if invalid_emails > 0:
        print(f"   - Examples: {df[~df['applicant_info.email'].str.contains('@', na=False)]['applicant_info.email'].head(3).tolist()}")

CHECKING SENSITIVE COLUMNS

 Column 'applicant_info.ssn' exists!
   - Number of non-null SSNs: 496
   - Sample values: ['596-64-4340', '425-69-4784', '370-78-5178']

 Column 'applicant_info.ip_address' exists!
   - Number of non-null IPs: 496
   - Sample values: ['192.168.48.155', '10.1.102.112', '10.240.193.250']

 Column 'applicant_info.email' exists (PII)
   - Number of non-null emails: 494
   - Emails without '@' symbol: 1
   - Examples: [nan, 'test.user.outlook.com', nan]


## 2. Pseudonymization Demonstration

Pseudonymization is the process of replacing identifiable data with artificial identifiers. 
Under GDPR (Recital 26), pseudonymized data is still considered personal data, but it significantly reduces privacy risks.

In this section, we demonstrate two pseudonymization techniques:

- **Hashing with salt** for email addresses (irreversible but consistent)
- **UUID tokenization** for Social Security Numbers (reversible with a mapping table)

We also create a sanitized dataset by removing direct identifiers while preserving analytical value.

### 2.1 Email Hashing with Salt

In [4]:

# PSEUDONYMIZATION DEMONSTRATION


print("PSEUDONYMIZATION DEMONSTRATION")

print("Method 1: Hashing with salt (for emails)")
print("Method 2: Tokenization with UUID (for SSNs)")


# Create a working copy
df_pseudo = df.copy()


# METHOD 1: HASHING WITH SALT (for emails)

print("\n EMAIL PSEUDONYMIZATION (SHA-256 + salt)")

def hash_email(email, salt="novacred_salt_2024"):
    """Pseudonymize email using SHA-256 with a secret salt"""
    if pd.isna(email):
        return email
    # Add salt to make hashing more secure (prevents rainbow table attacks)
    salted_email = str(email) + salt
    return hashlib.sha256(salted_email.encode('utf-8')).hexdigest()

# Show original emails
print("\n Original emails (sample):")
print(df_pseudo['applicant_info.email'].head(5).tolist())

# Apply hashing
df_pseudo['email_hashed'] = df_pseudo['applicant_info.email'].apply(hash_email)

# Show hashed emails
print("\n Hashed emails (sample):")
print(df_pseudo['email_hashed'].head(5).tolist())




PSEUDONYMIZATION DEMONSTRATION
Method 1: Hashing with salt (for emails)
Method 2: Tokenization with UUID (for SSNs)

 EMAIL PSEUDONYMIZATION (SHA-256 + salt)

 Original emails (sample):
['jerry.smith17@hotmail.com', 'brandon.walker2@yahoo.com', 'scott.moore94@mail.com', 'thomas.lee6@protonmail.com', 'brian.rodriguez86@aol.com']

 Hashed emails (sample):
['28b2cd139a84f1c7067ccca69027d5cc2002e1bf47d97b9b6fb0a1919e77ecf9', '1e46cdf120ca245230a82b63ed475a7b7c6516caa3e0b78bea3170463eb10e4c', '125e05986e4225a0ac55c21f692082f2089ceef26ec0b637876043e897c23c25', '919de8110fd1365715828095cef66c5bbdee5d70083f6ba3379b8a61062ce030', 'a9a413fd146e385ce7ff964c0369e32ef7c47a018d7f7a01ce05a5aec39c4d1d']


### 2.2 SSN Tokenization with UUID

In [ ]:
# METHOD 2: TOKENIZATION WITH UUID (for SSNs)


print(" SSN PSEUDONYMIZATION (UUID tokenization)")


import uuid

# Create a mapping dictionary to store the relationship between original SSN and token
# In production, this mapping would be stored in a secure, separate location
ssn_token_map = {}

def tokenize_ssn(ssn):
    """Replace SSN with a random UUID token"""
    if pd.isna(ssn):
        return ssn
    ssn_str = str(ssn)
    if ssn_str not in ssn_token_map:
        # Generate a new UUID for each unique SSN
        ssn_token_map[ssn_str] = str(uuid.uuid4())
    return ssn_token_map[ssn_str]

# Show original SSNs
print("\n Original SSNs (sample):")
print(df_pseudo['applicant_info.ssn'].head(5).tolist())

# Apply tokenization
df_pseudo['ssn_token'] = df_pseudo['applicant_info.ssn'].apply(tokenize_ssn)

# Show tokens
print("\n Tokenized SSNs (sample):")
print(df_pseudo['ssn_token'].head(5).tolist())




### 2.3 Sanitized Dataset

In [ ]:
# CREATE SANITIZED DATASET (remove original PII)


print(" CREATING SANITIZED DATASET")


# Keep only non-sensitive columns + pseudonymized versions
columns_to_drop = [
    'applicant_info.full_name',
    'applicant_info.email', 
    'applicant_info.ssn'
]

# Only drop columns that exist
columns_to_drop = [col for col in columns_to_drop if col in df_pseudo.columns]

df_sanitized = df_pseudo.drop(columns=columns_to_drop)

print(f"\n Sanitized dataset created!")
print(f"   - Original columns: {df_pseudo.shape[1]}")
print(f"   - Sanitized columns: {df_sanitized.shape[1]}")
print(f"\nColumns in sanitized dataset:")
print(df_sanitized.columns.tolist())



**Important note:** Under the General Data Protection Regulation (GDPR), it is important to be aware that pseudonymized data remains personal data. Although we have substituted direct identifiers such as emails and SSNs for hashes and tokens, there is nothing that stops the information from being correlated with the data subject if one were to get hold of the mapping table. This is also why the table of mappings that corresponds to the original SSNs and their UUID tokens must be stored separately for example on a different physical device than the pseudonymized dataset, with strict access control, and everything should be auditable, so if anyone should gain access to the table, everyone else is made aware of it. For in- depth analysis and model building, one should use the de-identified dataset to reduce privacy risk and keep the raw PII secure with limited access. This is a trade-off of utility and protection but it doesn’t relieve NovaCred of its GDPR obligations


## 3. GDPR Compliance Analysis

### 3.1 Article 5(1)(c) Data Minimisation

> "Personal data shall be adequate, relevant and limited to what is necessary in relation to the purposes for which they are processed." 
> — [GDPR Article 5(1)(c)](https://gdpr-info.eu/art-5-gdpr/#1-c)

The data minimisation principle requires that companies only process the absolute minimum of data necessary to complete the primary purpose of the data collection, i.e the initial purpose of the data collection. Data controllers need to carefully assess, based on [European Data Protection Board (EDPB)](https://edpb.europa.eu/our-work-tools/general-guidance/gdpr-guidelines-recommendations-best-practices_en),whether processing certain personal data is really necessary for achieving the intended purpose.

**Issue Identified:** The presence of *applicant_info.ssn* (Social Security Number) and *applicant_info.ip_address* in the dataset. Collecting SSNs for credit scoring is not only excessive but also unnecessary. Hence, this practice breaches the principle of data minimisation.

**Evidence:** 
- SSN column is present and has 496 non-null values
- IP address column is present and has 496 non-null values
- Absent a business justification for collecting such sensitive data

### 3.2 Article 5(1)(e) Storage Limitation

> "Personal data shall be kept in a form which permits identification of data subjects for no longer than is necessary for the purposes for which the personal data are processed." 
> — [GDPR Article 5(1)(e)](https://gdpr-info.eu/art-5-gdpr/#1-e)

According to the [European Data Protection Board (EDPB) guidelines on data protection principles](https://edpb.europa.eu/our-work-tools/general-guidance/gdpr-guidelines-recommendations-best-practices_en), organisations must define storage times before processing any data, ensuring that processing operations do not continue indefinitely once initial objectives are fulfilled.

**Issue Identified:** The dataset contains no field indicating retention period or expiry date. Data appears to be stored indefinitely without a clear deletion policy.

**Evidence:** No columns like *retention_date*, *expiry_timestamp*, or *data_deletion_schedule*.

### 3.3 Article 6 Lawfulness of Processing

> "Processing shall be lawful only if and to the extent that at least one of the following applies: (a) the data subject has given consent to the processing of his or her personal data for one or more specific purposes." 
> — [GDPR Article 6(1)(a)](https://gdpr-info.eu/art-6-gdpr/#1-a)

Where processing is based on consent, the controller must be able to demonstrate that the data subject has consented to processing of his or her personal data, as explicitly required by [GDPR Article 7(1)](https://gdpr-info.eu/art-7-gdpr/#1). According to the [European Data Protection Board (EDPB) guidelines on consent](https://edpb.europa.eu/our-work-tools/general-guidance/edpb-guidelines-052020-consent_en), consent must be freely given, specific, informed and unambiguous, and must be documented.

**Issue Identified:** The dataset contains no fields tracking user consent. There is no *consent_timestamp*, *consent_version*, or *consent_ip* to prove that applicants agreed to data processing.

**Evidence:** No consent-related columns in the dataset.

### 3.4 Article 17 Right to Erasure ("Right to be Forgotten")

> "The data subject shall have the right to obtain from the controller the erasure of personal data concerning him or her without undue delay... where one of the following grounds applies: (a) the personal data are no longer necessary in relation to the purposes for which they were collected or otherwise processed." 
> — [GDPR Article 17(1)(a)](https://gdpr-info.eu/art-17-gdpr/)

According to the [European Data Protection Board (EDPB) guidelines on the right to erasure](https://edpb.europa.eu/our-work-tools/our-documents/guidelines/guidelines-052019-processing-personal-data-context-video), controllers have an obligation to respond to erasure requests without undue delay and must have mechanisms in place to identify data subjects and process their requests.

**Issue Identified:** Invalid email addresses make it impossible to contact users to fulfil erasure requests. The controller has an obligation to erase personal data without undue delay when grounds apply, but cannot do so if the data subject cannot be identified or contacted.

**Evidence:** 
- 1 email without '@' symbol: *test.user.outlook.com*
- This user cannot be reached to delete their data if requested
- No alternative contact methods available (phone, mailing address)

### 3.5 Article 32 Security of Processing

> "Taking into account the state of the art... the controller and the processor shall implement appropriate technical and organisational measures to ensure a level of security appropriate to the risk, including inter alia as appropriate: (a) the pseudonymisation and encryption of personal data." 
> — [GDPR Article 32(1)(a)](https://gdpr-info.eu/art-32-gdpr/)

The [European Union Agency for Cybersecurity (ENISA)](https://www.enisa.europa.eu/publications/pseudonymisation-techniques-and-best-practices) recognises pseudonymisation as a key technical measure for data protection, stating that it "reduces the linkability of a dataset with the original identity of a data subject" while maintaining analytical value.

**Positive Finding:** The pseudonymisation demonstrated in Section 2 (hashing with salt for emails + UUID tokenization for SSNs) is an appropriate technical measure explicitly recognised by Article 32 as a means to ensure security of processing. The use of separate techniques for different data types demonstrates a risk-based approach to data protection.

**Implementation Note:** However, in production, the mapping table (*ssn_token_map*) must be stored separately from the pseudonymised dataset, with strict access controls, encryption at rest, and comprehensive access logging to comply with Article 32 requirements for "appropriate technical and organisational measures."

## 4. EU AI Act Analysis

Credit scoring systems are classified as **High-Risk AI Systems** under Annex III of the EU AI Act.

### 4.1 Classification

According to the [EU AI Act](https://artificialintelligenceact.eu/), [Article 6](https://artificialintelligenceact.eu/article/6/) and [Annex III, point 5(b)](https://artificialintelligenceact.eu/annex/3/), AI systems used to evaluate the creditworthiness of natural persons are considered high-risk. This means NovaCred's automated credit decision system falls under the strictest regulatory requirements.

### 4.2 Requirements for High-Risk AI Systems

| Requirement | What is missing | Source |
|-------------|-----------------|--------|
| **Article 10 – Data Governance** | Training data must be relevant, representative, and error-free. We found invalid values (negative credit history, DTI > 1) and missing values. | [AI Act Art. 10](https://artificialintelligenceact.eu/article/10/) |
| **Article 13 – Transparency** | Users must be informed they are interacting with an AI. Rejection reasons are generic (*algorithm_risk_score*) and not explainable. | [AI Act Art. 13](https://artificialintelligenceact.eu/article/13/) |
| **Article 14 – Human Oversight** | No evidence of human review for automated decisions. All rejections appear to be fully automated. | [AI Act Art. 14](https://artificialintelligenceact.eu/article/14/) |
| **Article 11 – Technical Documentation** | No documentation trail for model versions, training data, or bias mitigation measures. | [AI Act Art. 11](https://artificialintelligenceact.eu/article/11/) |

### 4.3 Consequences of Non-Compliance

> "Non-compliance with the requirements for high-risk AI systems shall be subject to administrative fines of up to €30,000,000 or, if the offender is a company, up to 6% of its total worldwide annual turnover for the preceding financial year, whichever is higher."
> — [AI Act, Article 99](https://artificialintelligenceact.eu/article/99/)

### 4.4 Summary

NovaCred's current AI system for credit scoring is operating as a high-risk AI system without meeting the mandatory requirements for data governance, transparency, human oversight, or technical documentation. This exposes the company to significant regulatory and financial risk.

## 5. Governance Recommendations 

The following recommendations are based on the specific data quality issues, privacy gaps, and regulatory non‑compliance identified in Sections 1–4. They are prioritised by urgency and impact.

### 5.1 Immediate Actions Short-term

| Priority | Recommendation | Rationale | Legal Basis |
|----------|----------------|-----------|-------------|
|  **High** | **Stop collecting SSN and IP address** | The dataset already contains 496 SSNs and IPs – collecting them is excessive for credit scoring and creates unnecessary risk. This directly violates the data minimisation principle. | [GDPR Art. 5(1)(c)](https://gdpr-info.eu/art-5-gdpr/#1-c) |
|  **High** | **Implement explicit consent mechanism** | There is currently no record of user consent. Add *consent_timestamp*, *consent_version*, and *consent_ip* to every application to demonstrate lawful processing. | [GDPR Art. 6(1)(a)](https://gdpr-info.eu/art-6-gdpr/#1-a), [Art. 7](https://gdpr-info.eu/art-7-gdpr/) |
|  **Medium** | **Fix input validation** | We found negative credit history values and a debt‑to‑income ratio above 1. These should be prevented at the point of entry (both frontend and backend). | [AI Act Art. 10](https://artificialintelligenceact.eu/article/10/) |

### 5.2 Medium‑term Actions 

| Priority | Recommendation | Rationale | Legal Basis |
|----------|----------------|-----------|-------------|
|  **Medium** | **Data retention policy** | Define max retention period (e.g., 3 years). Implement automatic deletion/anonymisation. | [GDPR Art. 5(1)(e)](https://gdpr-info.eu/art-5-gdpr/#1-e) |
|  **Medium** | **Audit trail (logging)** | Log every decision: application ID, timestamp, model version, input data, output decision | [AI Act Art. 12](https://artificialintelligenceact.eu/article/12/) |
|  **Medium** | **Human-in-the-loop** | Review all rejected applications manually, especially cases flagged by bias analysis | [AI Act Art. 14](https://artificialintelligenceact.eu/article/14/) |

### 5.3 Long-term Actions 

| Priority | Recommendation | Rationale | Legal Basis |
|----------|----------------|-----------|-------------|
| **Low** | **Bias monitoring** | Quarterly fairness reports (Disparate Impact Ratio by gender, age). Retrain if bias exceeds thresholds. | [AI Act Art. 10](https://artificialintelligenceact.eu/article/10/), [Art. 15](https://artificialintelligenceact.eu/article/15/) |
| **Low** | **DPIA (Data Protection Impact Assessment)** | Conduct formal DPIA as required for high-risk processing | [GDPR Art. 35](https://gdpr-info.eu/art-35-gdpr/) |
| **Low** | **Explainable AI** | Replace generic rejection reasons with specific explanations (e.g., "Credit history too short") | [AI Act Art. 13](https://artificialintelligenceact.eu/article/13/) |

### 5.4 Technical Implementation Priorities

1. **Immediately** – Remove SSN and IP fields from the application form and stop storing them.
2. **Immediately** – Add a consent checkbox and store the consent proof (timestamp, version, IP).
3. **Short‑term** – Implement frontend validation to reject negative credit history and DTI > 1 at submission.
4. **Medium‑term** – Develop scripts to automatically delete or anonymise records older than the retention period.
5. **Medium‑term** – Build a logging system that records every decision for audit purposes.
6. **Long‑term** – Integrate fairness metrics into the model monitoring pipeline and schedule regular reviews.